<a href="https://colab.research.google.com/github/william-idam-inya/Group-26_Student-Academic-Outcome-Prediction_Notebook.ipynb/blob/main/SOP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q streamlit pandas numpy matplotlib scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 52.3 MB/s eta 0:00:00


In [3]:
import pandas as pd
import os

DATA_PATH = "/content/drive/MyDrive/student-performance-app/data.csv"

print("Data file exists:", os.path.exists(DATA_PATH))

df = pd.read_csv(
    DATA_PATH,
    sep=";"
)

# Clean column names
df.columns = (
    df.columns
    .astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.replace("\t", "", regex=False)
    .str.strip()
)

# Correct spelling used in the original dataset
if "Nacionality" in df.columns:
    df.rename(
        columns={
            "Nacionality": "Nationality"
        },
        inplace=True
    )

print("\nDataset shape:", df.shape)

print("\nColumn names:")
for i, column in enumerate(df.columns, start=1):
    print(i, column)

print("\nTarget distribution:")
print(df["Target"].value_counts())

print("\nFirst 5 rows:")
display(df.head())

Data file exists: True

Dataset shape: (4424, 37)

Column names:
1 Marital status
2 Application mode
3 Application order
4 Course
5 Daytime/evening attendance
6 Previous qualification
7 Previous qualification (grade)
8 Nationality
9 Mother's qualification
10 Father's qualification
11 Mother's occupation
12 Father's occupation
13 Admission grade
14 Displaced
15 Educational special needs
16 Debtor
17 Tuition fees up to date
18 Gender
19 Scholarship holder
20 Age at enrollment
21 International
22 Curricular units 1st sem (credited)
23 Curricular units 1st sem (enrolled)
24 Curricular units 1st sem (evaluations)
25 Curricular units 1st sem (approved)
26 Curricular units 1st sem (grade)
27 Curricular units 1st sem (without evaluations)
28 Curricular units 2nd sem (credited)
29 Curricular units 2nd sem (enrolled)
30 Curricular units 2nd sem (evaluations)
31 Curricular units 2nd sem (approved)
32 Curricular units 2nd sem (grade)
33 Curricular units 2nd sem (without evaluations)
34 Unemploymen

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nationality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [4]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

!/content/cloudflared --version

cloudflared version 2026.9.3 (built 2026-09-24-16:07 UTC)


In [5]:
%%writefile /content/drive/MyDrive/student-performance-app/app.py

import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Student Performance Predictor",
    page_icon="🎓",
    layout="wide"
)


# ============================================================
# CUSTOM STYLING
# ============================================================

st.markdown(
    """
    <style>

    .main-title {
        font-size: 42px;
        font-weight: 700;
        margin-bottom: 5px;
    }

    .subtitle {
        font-size: 18px;
        color: #666666;
        margin-bottom: 25px;
    }

    .prediction-box {
        padding: 30px;
        border-radius: 15px;
        border: 2px solid #d0d7de;
        text-align: center;
        margin-top: 20px;
        margin-bottom: 20px;
    }

    .prediction-result {
        font-size: 42px;
        font-weight: 700;
        margin-top: 10px;
    }

    .section-description {
        color: #666666;
        margin-bottom: 15px;
    }

    </style>
    """,
    unsafe_allow_html=True
)


# ============================================================
# DATA PATH
# ============================================================

DATA_PATH = (
    "/content/drive/MyDrive/"
    "student-performance-app/data.csv"
)


# ============================================================
# LOAD DATA
# ============================================================

@st.cache_data
def load_data():

    data = pd.read_csv(
        DATA_PATH,
        sep=";"
    )

    # Clean column names
    data.columns = (
        data.columns
        .astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.replace("\t", "", regex=False)
        .str.strip()
    )

    # Correct the spelling used by the source dataset
    if "Nacionality" in data.columns:

        data.rename(
            columns={
                "Nacionality": "Nationality"
            },
            inplace=True
        )

    return data


# ============================================================
# LOAD DATASET
# ============================================================

try:

    df = load_data()

except Exception as error:

    st.error(
        "❌ The dataset could not be loaded."
    )

    st.exception(error)

    st.stop()


# ============================================================
# CHECK TARGET
# ============================================================

if "Target" not in df.columns:

    st.error(
        "❌ The Target column was not found."
    )

    st.write(
        "Columns detected:"
    )

    st.write(
        df.columns.tolist()
    )

    st.stop()


# Remove rows with missing target
df = df.dropna(
    subset=["Target"]
).reset_index(drop=True)


# ============================================================
# FEATURES AND TARGET
# ============================================================

X = df.drop(
    "Target",
    axis=1
)

y = df["Target"]


# ============================================================
# NUMERICAL FEATURES
# ============================================================

numerical_features = [

    "Previous qualification (grade)",

    "Admission grade",

    "Age at enrollment",

    "Curricular units 1st sem (credited)",

    "Curricular units 1st sem (enrolled)",

    "Curricular units 1st sem (evaluations)",

    "Curricular units 1st sem (approved)",

    "Curricular units 1st sem (grade)",

    "Curricular units 1st sem (without evaluations)",

    "Curricular units 2nd sem (credited)",

    "Curricular units 2nd sem (enrolled)",

    "Curricular units 2nd sem (evaluations)",

    "Curricular units 2nd sem (approved)",

    "Curricular units 2nd sem (grade)",

    "Curricular units 2nd sem (without evaluations)",

    "Unemployment rate",

    "Inflation rate",

    "GDP"
]


# ============================================================
# CATEGORICAL FEATURES
# ============================================================

categorical_features = [

    column

    for column in X.columns

    if column not in numerical_features

]


# ============================================================
# HUMAN-READABLE DISPLAY NAMES
# ============================================================

DISPLAY_NAMES = {

    "Marital status":
        "Marital Status",

    "Application mode":
        "Application Mode",

    "Application order":
        "Application Order",

    "Course":
        "Course",

    "Daytime/evening attendance":
        "Attendance Type",

    "Previous qualification":
        "Previous Qualification",

    "Previous qualification (grade)":
        "Previous Qualification Grade",

    "Nationality":
        "Nationality",

    "Mother's qualification":
        "Mother's Qualification",

    "Father's qualification":
        "Father's Qualification",

    "Mother's occupation":
        "Mother's Occupation",

    "Father's occupation":
        "Father's Occupation",

    "Admission grade":
        "Admission Grade",

    "Displaced":
        "Displaced Student",

    "Educational special needs":
        "Educational Special Needs",

    "Debtor":
        "Debtor",

    "Tuition fees up to date":
        "Tuition Fees Up to Date",

    "Gender":
        "Gender",

    "Scholarship holder":
        "Scholarship Holder",

    "Age at enrollment":
        "Age at Enrollment",

    "International":
        "International Student",

    "Curricular units 1st sem (credited)":
        "1st Semester - Credited Units",

    "Curricular units 1st sem (enrolled)":
        "1st Semester - Enrolled Units",

    "Curricular units 1st sem (evaluations)":
        "1st Semester - Evaluations",

    "Curricular units 1st sem (approved)":
        "1st Semester - Approved Units",

    "Curricular units 1st sem (grade)":
        "1st Semester - Grade",

    "Curricular units 1st sem (without evaluations)":
        "1st Semester - Units Without Evaluation",

    "Curricular units 2nd sem (credited)":
        "2nd Semester - Credited Units",

    "Curricular units 2nd sem (enrolled)":
        "2nd Semester - Enrolled Units",

    "Curricular units 2nd sem (evaluations)":
        "2nd Semester - Evaluations",

    "Curricular units 2nd sem (approved)":
        "2nd Semester - Approved Units",

    "Curricular units 2nd sem (grade)":
        "2nd Semester - Grade",

    "Curricular units 2nd sem (without evaluations)":
        "2nd Semester - Units Without Evaluation",

    "Unemployment rate":
        "Unemployment Rate (%)",

    "Inflation rate":
        "Inflation Rate (%)",

    "GDP":
        "GDP"
}


# ============================================================
# HUMAN-READABLE CATEGORY MAPPINGS
# ============================================================

DISPLAY_MAPS = {

    # --------------------------------------------------------
    # MARITAL STATUS
    # --------------------------------------------------------

    "Marital status": {

        1: "Single",
        2: "Married",
        3: "Widower",
        4: "Divorced",
        5: "Facto Union",
        6: "Legally Separated"

    },


    # --------------------------------------------------------
    # APPLICATION MODE
    # --------------------------------------------------------

    "Application mode": {

        1: "1st Phase - General Contingent",

        2: "Ordinance No. 612/93",

        5: "1st Phase - Special Contingent (Azores Island)",

        7: "Holders of Other Higher Courses",

        10: "Ordinance No. 854-B/99",

        15: "International Student (Bachelor)",

        16: "1st Phase - Special Contingent (Madeira Island)",

        17: "2nd Phase - General Contingent",

        18: "3rd Phase - General Contingent",

        26: "Ordinance No. 533-A/99 - Different Plan",

        27: "Ordinance No. 533-A/99 - Other Institution",

        39: "Over 23 Years Old",

        42: "Transfer",

        43: "Change of Course",

        44: "Technological Specialization Diploma Holders",

        51: "Change of Institution/Course",

        53: "Short Cycle Diploma Holders",

        57: "Change of Institution/Course - International"

    },


    # --------------------------------------------------------
    # APPLICATION ORDER
    # --------------------------------------------------------

    "Application order": {

        0: "1st Choice",

        1: "2nd Choice",

        2: "3rd Choice",

        3: "4th Choice",

        4: "5th Choice",

        5: "6th Choice",

        6: "7th Choice",

        7: "8th Choice",

        8: "9th Choice",

        9: "10th Choice"

    },


    # --------------------------------------------------------
    # COURSE
    # --------------------------------------------------------

    "Course": {

        33: "Biofuel Production Technologies",

        171: "Animation and Multimedia Design",

        8014: "Social Service (Evening Attendance)",

        9003: "Agronomy",

        9070: "Communication Design",

        9085: "Veterinary Nursing",

        9119: "Informatics Engineering",

        9130: "Equinculture",

        9147: "Management",

        9238: "Social Service",

        9254: "Tourism",

        9500: "Nursing",

        9556: "Oral Hygiene",

        9670: "Advertising and Marketing Management",

        9773: "Journalism and Communication",

        9853: "Basic Education",

        9991: "Management (Evening Attendance)"

    },


    # --------------------------------------------------------
    # ATTENDANCE
    # --------------------------------------------------------

    "Daytime/evening attendance": {

        0: "Evening",

        1: "Daytime"

    },


    # --------------------------------------------------------
    # PREVIOUS QUALIFICATION
    # --------------------------------------------------------

    "Previous qualification": {

        1: "Secondary Education",

        2: "Higher Education - Bachelor's Degree",

        3: "Higher Education - Degree",

        4: "Higher Education - Master's",

        5: "Higher Education - Doctorate",

        6: "Frequency of Higher Education",

        9: "12th Year of Schooling - Not Completed",

        10: "11th Year of Schooling - Not Completed",

        12: "Other - 11th Year of Schooling",

        14: "10th Year of Schooling",

        15: "10th Year of Schooling - Not Completed",

        19: "Basic Education 3rd Cycle (9th/10th/11th Year) or Equivalent",

        38: "Basic Education 2nd Cycle (6th/7th/8th Year) or Equivalent",

        39: "Technological Specialization Course",

        40: "Higher Education - Degree (1st Cycle)",

        42: "Professional Higher Technical Course",

        43: "Higher Education - Master (2nd Cycle)"

    },


    # --------------------------------------------------------
    # NATIONALITY
    # --------------------------------------------------------

    "Nationality": {

        1: "Portuguese",

        2: "German",

        6: "Spanish",

        11: "Italian",

        13: "Dutch",

        14: "English",

        17: "Lithuanian",

        21: "Angolan",

        22: "Cape Verdean",

        24: "Guinean",

        25: "Mozambican",

        26: "Santomean",

        32: "Turkish",

        41: "Brazilian",

        62: "Romanian",

        100: "Moldova (Republic of)",

        101: "Mexican",

        103: "Ukrainian",

        105: "Russian",

        108: "Cuban",

        109: "Colombian"

    },


    # --------------------------------------------------------
    # MOTHER'S QUALIFICATION
    # --------------------------------------------------------

    "Mother's qualification": {

        1: "Secondary Education - 12th Year or Equivalent",

        2: "Higher Education - Bachelor's Degree",

        3: "Higher Education - Degree",

        4: "Higher Education - Master's",

        5: "Higher Education - Doctorate",

        6: "Frequency of Higher Education",

        9: "12th Year of Schooling - Not Completed",

        10: "11th Year of Schooling - Not Completed",

        11: "7th Year (Old)",

        12: "Other - 11th Year of Schooling",

        14: "10th Year of Schooling",

        18: "General Commerce Course",

        19: "Basic Education 3rd Cycle (9th/10th/11th Year) or Equivalent",

        22: "Technical-Professional Course",

        26: "7th Year of Schooling",

        27: "2nd Cycle of the General High School Course",

        29: "9th Year of Schooling - Not Completed",

        30: "8th Year of Schooling",

        34: "Unknown",

        35: "Cannot Read or Write",

        36: "Can Read Without Having a 4th Year of Schooling",

        37: "Basic Education 1st Cycle (4th/5th Year) or Equivalent",

        38: "Basic Education 2nd Cycle (6th/7th/8th Year) or Equivalent",

        39: "Technological Specialization Course",

        40: "Higher Education - Degree (1st Cycle)",

        41: "Specialized Higher Studies Course",

        42: "Professional Higher Technical Course",

        43: "Higher Education - Master (2nd Cycle)",

        44: "Higher Education - Doctorate (3rd Cycle)"

    },


    # --------------------------------------------------------
    # FATHER'S QUALIFICATION
    # --------------------------------------------------------

    "Father's qualification": {

        1: "Secondary Education - 12th Year or Equivalent",

        2: "Higher Education - Bachelor's Degree",

        3: "Higher Education - Degree",

        4: "Higher Education - Master's",

        5: "Higher Education - Doctorate",

        6: "Frequency of Higher Education",

        9: "12th Year of Schooling - Not Completed",

        10: "11th Year of Schooling - Not Completed",

        11: "7th Year (Old)",

        12: "Other - 11th Year of Schooling",

        13: "2nd Year Complementary High School Course",

        14: "10th Year of Schooling",

        18: "General Commerce Course",

        19: "Basic Education 3rd Cycle (9th/10th/11th Year) or Equivalent",

        20: "Complementary High School Course",

        22: "Technical-Professional Course",

        25: "Complementary High School Course - Not Concluded",

        26: "7th Year of Schooling",

        27: "2nd Cycle of the General High School Course",

        29: "9th Year of Schooling - Not Completed",

        30: "8th Year of Schooling",

        31: "General Course of Administration and Commerce",

        33: "Supplementary Accounting and Administration",

        34: "Unknown",

        35: "Cannot Read or Write",

        36: "Can Read Without Having a 4th Year of Schooling",

        37: "Basic Education 1st Cycle (4th/5th Year) or Equivalent",

        38: "Basic Education 2nd Cycle (6th/7th/8th Year) or Equivalent",

        39: "Technological Specialization Course",

        40: "Higher Education - Degree (1st Cycle)",

        41: "Specialized Higher Studies Course",

        42: "Professional Higher Technical Course",

        43: "Higher Education - Master (2nd Cycle)",

        44: "Higher Education - Doctorate (3rd Cycle)"

    },


    # --------------------------------------------------------
    # MOTHER'S OCCUPATION
    # --------------------------------------------------------

    "Mother's occupation": {

        0: "Student",

        1: "Legislative/Executive Representative or Executive Manager",

        2: "Specialist in Intellectual and Scientific Activities",

        3: "Intermediate-Level Technician or Professional",

        4: "Administrative Staff",

        5: "Personal Services, Security/Safety Worker or Seller",

        6: "Farmer or Skilled Agricultural/Forestry/Fisheries Worker",

        7: "Skilled Worker in Industry, Construction or Craft",

        8: "Machine Operator or Assembly Worker",

        9: "Unskilled Worker",

        10: "Armed Forces Professional",

        90: "Other Situation",

        99: "Not Specified",

        122: "Health Professional",

        123: "Teacher",

        125: "ICT Specialist",

        131: "Science/Engineering Intermediate Technician",

        132: "Intermediate Health Professional",

        134: "Intermediate Legal/Social/Sports/Cultural Professional",

        141: "Office Worker/Secretary/Data Processing Operator",

        143: "Accounting/Statistical/Financial Services Operator",

        144: "Other Administrative Support Staff",

        151: "Personal Service Worker",

        152: "Seller",

        153: "Personal Care Worker",

        171: "Skilled Construction Worker",

        173: "Skilled Printing/Precision Instrument/Jewelry/Artisan Worker",

        175: "Food Processing/Woodworking/Clothing/Industry Worker",

        191: "Cleaning Worker",

        192: "Unskilled Agricultural/Animal/Fisheries/Forestry Worker",

        193: "Unskilled Extractive/Construction/Manufacturing/Transport Worker",

        194: "Meal Preparation Assistant"

    },


    # --------------------------------------------------------
    # FATHER'S OCCUPATION
    # --------------------------------------------------------

    "Father's occupation": {

        0: "Student",

        1: "Legislative/Executive Representative or Executive Manager",

        2: "Specialist in Intellectual and Scientific Activities",

        3: "Intermediate-Level Technician or Professional",

        4: "Administrative Staff",

        5: "Personal Services, Security/Safety Worker or Seller",

        6: "Farmer or Skilled Agricultural/Forestry/Fisheries Worker",

        7: "Skilled Worker in Industry, Construction or Craft",

        8: "Machine Operator or Assembly Worker",

        9: "Unskilled Worker",

        10: "Armed Forces Professional",

        90: "Other Situation",

        99: "Not Specified",

        101: "Armed Forces Officer",

        102: "Armed Forces Sergeant",

        103: "Other Armed Forces Personnel",

        112: "Director of Administrative and Commercial Services",

        114: "Hotel/Catering/Trade/Other Services Director",

        121: "Physical Science/Mathematics/Engineering Specialist",

        122: "Health Professional",

        123: "Teacher",

        124: "Finance/Accounting/Administrative/Public Relations Specialist",

        131: "Science/Engineering Intermediate Technician",

        132: "Intermediate Health Professional",

        134: "Intermediate Legal/Social/Sports/Cultural Professional",

        135: "ICT Technician",

        141: "Office Worker/Secretary/Data Processing Operator",

        143: "Accounting/Statistical/Financial Services Operator",

        144: "Other Administrative Support Staff",

        151: "Personal Service Worker",

        152: "Seller",

        153: "Personal Care Worker",

        154: "Protection and Security Services Personnel",

        161: "Market-Oriented Farmer/Agricultural/Animal Production Worker",

        163: "Subsistence Farmer/Livestock Keeper/Fisher/Hunter/Gatherer",

        171: "Skilled Construction Worker",

        172: "Skilled Metallurgy/Metalworking Worker",

        174: "Skilled Electricity/Electronics Worker",

        175: "Food Processing/Woodworking/Clothing/Industry Worker",

        181: "Fixed Plant and Machine Operator",

        182: "Assembly Worker",

        183: "Vehicle Driver/Mobile Equipment Operator",

        192: "Unskilled Agricultural/Animal/Fisheries/Forestry Worker",

        193: "Unskilled Extractive/Construction/Manufacturing/Transport Worker",

        194: "Meal Preparation Assistant",

        195: "Street Vendor/Street Service Provider"

    },


    # --------------------------------------------------------
    # BOOLEAN VARIABLES
    # --------------------------------------------------------

    "Displaced": {

        0: "No",
        1: "Yes"

    },

    "Educational special needs": {

        0: "No",
        1: "Yes"

    },

    "Debtor": {

        0: "No",
        1: "Yes"

    },

    "Tuition fees up to date": {

        0: "No",
        1: "Yes"

    },

    "Gender": {

        0: "Female",
        1: "Male"

    },

    "Scholarship holder": {

        0: "No",
        1: "Yes"

    },

    "International": {

        0: "No",
        1: "Yes"

    }

}


# ============================================================
# MODEL TRAINING
# ============================================================

@st.cache_resource
def train_model(data):

    X_data = data.drop(
        "Target",
        axis=1
    )

    y_data = data["Target"]


    # --------------------------------------------------------
    # TRAIN / TEST SPLIT
    # --------------------------------------------------------

    X_train, X_test, y_train, y_test = train_test_split(

        X_data,

        y_data,

        test_size=0.20,

        random_state=42,

        stratify=y_data

    )


    # --------------------------------------------------------
    # NUMERICAL TRANSFORMATION
    # --------------------------------------------------------

    numerical_transformer = Pipeline(

        steps=[

            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),

            (
                "scaler",
                StandardScaler()
            )

        ]

    )


    # --------------------------------------------------------
    # CATEGORICAL TRANSFORMATION
    # --------------------------------------------------------

    categorical_transformer = Pipeline(

        steps=[

            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),

            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )

        ]

    )


    # --------------------------------------------------------
    # COMBINE TRANSFORMERS
    # --------------------------------------------------------

    preprocessor = ColumnTransformer(

        transformers=[

            (
                "num",
                numerical_transformer,
                numerical_features
            ),

            (
                "cat",
                categorical_transformer,
                categorical_features
            )

        ]

    )


    # --------------------------------------------------------
    # COMPLETE MODEL PIPELINE
    # --------------------------------------------------------

    model_pipeline = Pipeline(

        steps=[

            (
                "preprocessor",
                preprocessor
            ),

            (
                "model",
                LogisticRegression(
                    max_iter=1000,
                    random_state=42
                )
            )

        ]

    )


    # --------------------------------------------------------
    # TRAIN MODEL
    # --------------------------------------------------------

    model_pipeline.fit(
        X_train,
        y_train
    )


    # --------------------------------------------------------
    # TEST MODEL
    # --------------------------------------------------------

    y_pred = model_pipeline.predict(
        X_test
    )


    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    confusion = confusion_matrix(
        y_test,
        y_pred,
        labels=model_pipeline.classes_
    )

    report = classification_report(
        y_test,
        y_pred,
        zero_division=0
    )


    return (
        model_pipeline,
        accuracy,
        precision,
        recall,
        f1,
        confusion,
        report,
        len(X_train),
        len(X_test)
    )


# ============================================================
# TRAIN MODEL
# ============================================================

with st.spinner(
    "Preparing the student outcome prediction model..."
):

    (
        model_pipeline,
        accuracy,
        precision,
        recall,
        f1,
        confusion,
        report,
        train_size,
        test_size
    ) = train_model(df)


# ============================================================
# PAGE HEADER
# ============================================================

st.markdown(
    '<div class="main-title">'
    '🎓 Student Performance Predictor'
    '</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">'
    'Enter the student\'s information below to generate '
    'a predicted academic outcome.'
    '</div>',
    unsafe_allow_html=True
)


st.info(
    "The app uses the trained Logistic Regression model "
    "to predict the student's recorded outcome category. "
    "The result is a model prediction, not a guarantee of "
    "the student's future performance."
)


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    st.header(
        "📊 Model Information"
    )

    st.write(
        "**Algorithm:** Logistic Regression"
    )

    st.write(
        f"**Students:** {len(df):,}"
    )

    st.write(
        f"**Training set:** {train_size:,}"
    )

    st.write(
        f"**Test set:** {test_size:,}"
    )

    st.divider()

    st.subheader(
        "Test Set Performance"
    )

    st.metric(
        "Accuracy",
        f"{accuracy:.2%}"
    )

    st.metric(
        "Precision",
        f"{precision:.2%}"
    )

    st.metric(
        "Recall",
        f"{recall:.2%}"
    )

    st.metric(
        "F1 Score",
        f"{f1:.2%}"
    )


# ============================================================
# STUDENT NAME
# ============================================================

st.subheader(
    "👤 Student Information"
)

student_name = st.text_input(
    "Student Name",
    placeholder="Enter student's name"
)


# ============================================================
# PREDICTION FORM
# ============================================================

with st.form(
    "student_prediction_form"
):

    # ========================================================
    # SECTION 1 — PERSONAL / DEMOGRAPHIC
    # ========================================================

    st.markdown(
        "## 👤 Personal & Demographic Information"
    )

    st.caption(
        "Enter the student's personal and demographic information."
    )

    user_inputs = {}


    demographic_features = [

        "Marital status",

        "Nationality",

        "Gender",

        "Age at enrollment",

        "International",

        "Displaced",

        "Educational special needs"

    ]


    demographic_cat = [
        feature
        for feature in demographic_features
        if feature in categorical_features
    ]


    demographic_num = [
        feature
        for feature in demographic_features
        if feature in numerical_features
    ]


    # --------------------------------------------------------
    # Demographic categorical fields
    # --------------------------------------------------------

    cols = st.columns(2)

    for index, feature in enumerate(
        demographic_cat
    ):

        with cols[index % 2]:

            values = (
                X[feature]
                .dropna()
                .unique()
                .tolist()
            )

            values = sorted(values)

            mapping = DISPLAY_MAPS.get(
                feature,
                {}
            )

            if len(values) == 2:

                selected = st.radio(

                    DISPLAY_NAMES.get(
                        feature,
                        feature
                    ),

                    values,

                    format_func=lambda x:
                        mapping.get(
                            x,
                            str(x)
                        ),

                    horizontal=True

                )

            else:

                selected = st.selectbox(

                    DISPLAY_NAMES.get(
                        feature,
                        feature
                    ),

                    values,

                    format_func=lambda x:
                        mapping.get(
                            x,
                            str(x)
                        )

                )

            user_inputs[feature] = selected


    # --------------------------------------------------------
    # Age
    # --------------------------------------------------------

    if "Age at enrollment" in numerical_features:

        age_values = pd.to_numeric(
            X["Age at enrollment"],
            errors="coerce"
        )

        with st.columns(2)[0]:

            user_inputs[
                "Age at enrollment"
            ] = st.number_input(

                "Age at Enrollment",

                min_value=int(
                    age_values.min()
                ),

                max_value=int(
                    age_values.max()
                ),

                value=int(
                    age_values.median()
                ),

                step=1

            )


    # ========================================================
    # SECTION 2 — APPLICATION / EDUCATION
    # ========================================================

    st.divider()

    st.markdown(
        "## 🎓 Application & Educational Background"
    )

    st.caption(
        "Enter information about how the student entered the institution "
        "and their previous educational background."
    )


    education_categorical = [

        "Application mode",

        "Application order",

        "Course",

        "Daytime/evening attendance",

        "Previous qualification"

    ]


    cols = st.columns(2)


    for index, feature in enumerate(
        education_categorical
    ):

        with cols[index % 2]:

            values = (
                X[feature]
                .dropna()
                .unique()
                .tolist()
            )

            values = sorted(values)

            mapping = DISPLAY_MAPS.get(
                feature,
                {}
            )


            # Binary attendance
            if len(values) == 2:

                selected = st.radio(

                    DISPLAY_NAMES.get(
                        feature,
                        feature
                    ),

                    values,

                    format_func=lambda x:
                        mapping.get(
                            x,
                            str(x)
                        ),

                    horizontal=True

                )


            else:

                selected = st.selectbox(

                    DISPLAY_NAMES.get(
                        feature,
                        feature
                    ),

                    values,

                    format_func=lambda x:
                        mapping.get(
                            x,
                            str(x)
                        )

                )


            user_inputs[feature] = selected


    # --------------------------------------------------------
    # Previous qualification grade
    # --------------------------------------------------------

    previous_grade_values = pd.to_numeric(
        X[
            "Previous qualification (grade)"
        ],
        errors="coerce"
    )


    with st.columns(2)[0]:

        user_inputs[
            "Previous qualification (grade)"
        ] = st.number_input(

            "Previous Qualification Grade",

            min_value=float(
                previous_grade_values.min()
            ),

            max_value=float(
                previous_grade_values.max()
            ),

            value=float(
                previous_grade_values.median()
            ),

            step=0.1,

            format="%.1f"

        )


    # --------------------------------------------------------
    # Admission grade
    # --------------------------------------------------------

    admission_values = pd.to_numeric(
        X[
            "Admission grade"
        ],
        errors="coerce"
    )


    with st.columns(2)[1]:

        user_inputs[
            "Admission grade"
        ] = st.number_input(

            "Admission Grade",

            min_value=float(
                admission_values.min()
            ),

            max_value=float(
                admission_values.max()
            ),

            value=float(
                admission_values.median()
            ),

            step=0.1,

            format="%.1f"

        )


    # ========================================================
    # SECTION 3 — FAMILY BACKGROUND
    # ========================================================

    st.divider()

    st.markdown(
        "## 👨‍👩‍👧‍👦 Family Background"
    )

    family_features = [

        "Mother's qualification",

        "Father's qualification",

        "Mother's occupation",

        "Father's occupation"

    ]


    cols = st.columns(2)


    for index, feature in enumerate(
        family_features
    ):

        with cols[index % 2]:

            values = (
                X[feature]
                .dropna()
                .unique()
                .tolist()
            )

            values = sorted(values)

            mapping = DISPLAY_MAPS.get(
                feature,
                {}
            )


            selected = st.selectbox(

                DISPLAY_NAMES.get(
                    feature,
                    feature
                ),

                values,

                format_func=lambda x:
                    mapping.get(
                        x,
                        str(x)
                    )

            )


            user_inputs[feature] = selected


    # ========================================================
    # SECTION 4 — FINANCIAL / SUPPORT
    # ========================================================

    st.divider()

    st.markdown(
        "## 💳 Financial & Student Support Information"
    )


    financial_features = [

        "Debtor",

        "Tuition fees up to date",

        "Scholarship holder"

    ]


    cols = st.columns(3)


    for index, feature in enumerate(
        financial_features
    ):

        with cols[index]:

            values = (
                X[feature]
                .dropna()
                .unique()
                .tolist()
            )

            values = sorted(values)

            mapping = DISPLAY_MAPS.get(
                feature,
                {}
            )


            user_inputs[feature] = st.radio(

                DISPLAY_NAMES.get(
                    feature,
                    feature
                ),

                values,

                format_func=lambda x:
                    mapping.get(
                        x,
                        str(x)
                    ),

                horizontal=True

            )


    # ========================================================
    # SECTION 5 — FIRST SEMESTER
    # ========================================================

    st.divider()

    st.markdown(
        "## 📚 First Semester Academic Performance"
    )


    first_semester_features = [

        "Curricular units 1st sem (credited)",

        "Curricular units 1st sem (enrolled)",

        "Curricular units 1st sem (evaluations)",

        "Curricular units 1st sem (approved)",

        "Curricular units 1st sem (grade)",

        "Curricular units 1st sem (without evaluations)"

    ]


    cols = st.columns(2)


    for index, feature in enumerate(
        first_semester_features
    ):

        values = pd.to_numeric(
            X[feature],
            errors="coerce"
        )


        with cols[index % 2]:

            is_integer = (
                (values.dropna() % 1 == 0).all()
            )


            if is_integer:

                user_inputs[feature] = st.number_input(

                    DISPLAY_NAMES.get(
                        feature,
                        feature
                    ),

                    min_value=int(
                        values.min()
                    ),

                    max_value=int(
                        values.max()
                    ),

                    value=int(
                        values.median()
                    ),

                    step=1

                )

            else:

                user_inputs[feature] = st.number_input(

                    DISPLAY_NAMES.get(
                        feature,
                        feature
                    ),

                    min_value=float(
                        values.min()
                    ),

                    max_value=float(
                        values.max()
                    ),

                    value=float(
                        values.median()
                    ),

                    step=0.1,

                    format="%.1f"

                )


    # ========================================================
    # SECTION 6 — SECOND SEMESTER
    # ========================================================

    st.divider()

    st.markdown(
        "## 📚 Second Semester Academic Performance"
    )


    second_semester_features = [

        "Curricular units 2nd sem (credited)",

        "Curricular units 2nd sem (enrolled)",

        "Curricular units 2nd sem (evaluations)",

        "Curricular units 2nd sem (approved)",

        "Curricular units 2nd sem (grade)",

        "Curricular units 2nd sem (without evaluations)"

    ]


    cols = st.columns(2)


    for index, feature in enumerate(
        second_semester_features
    ):

        values = pd.to_numeric(
            X[feature],
            errors="coerce"
        )


        with cols[index % 2]:

            is_integer = (
                (values.dropna() % 1 == 0).all()
            )


            if is_integer:

                user_inputs[feature] = st.number_input(

                    DISPLAY_NAMES.get(
                        feature,
                        feature
                    ),

                    min_value=int(
                        values.min()
                    ),

                    max_value=int(
                        values.max()
                    ),

                    value=int(
                        values.median()
                    ),

                    step=1

                )

            else:

                user_inputs[feature] = st.number_input(

                    DISPLAY_NAMES.get(
                        feature,
                        feature
                    ),

                    min_value=float(
                        values.min()
                    ),

                    max_value=float(
                        values.max()
                    ),

                    value=float(
                        values.median()
                    ),

                    step=0.1,

                    format="%.1f"

                )


    # ========================================================
    # SECTION 7 — ECONOMIC CONTEXT
    # ========================================================

    st.divider()

    st.markdown(
        "## 🌍 Economic Context"
    )


    economic_features = [

        "Unemployment rate",

        "Inflation rate",

        "GDP"

    ]


    cols = st.columns(3)


    for index, feature in enumerate(
        economic_features
    ):

        values = pd.to_numeric(
            X[feature],
            errors="coerce"
        )


        with cols[index]:

            user_inputs[feature] = st.number_input(

                DISPLAY_NAMES.get(
                    feature,
                    feature
                ),

                min_value=float(
                    values.min()
                ),

                max_value=float(
                    values.max()
                ),

                value=float(
                    values.median()
                ),

                step=0.01,

                format="%.2f"

            )


    # ========================================================
    # PREDICT BUTTON
    # ========================================================

    st.divider()

    submitted = st.form_submit_button(

        "🔮 PREDICT STUDENT OUTCOME",

        use_container_width=True

    )


# ============================================================
# MAKE PREDICTION
# ============================================================

if submitted:

    try:

        # ----------------------------------------------------
        # Create one-row dataframe
        # ----------------------------------------------------

        input_df = pd.DataFrame(
            [user_inputs]
        )


        # Make sure the columns are in exactly the same
        # order as the training data
        input_df = input_df[
            X.columns
        ]


        # ----------------------------------------------------
        # Predict
        # ----------------------------------------------------

        prediction = model_pipeline.predict(
            input_df
        )[0]


        # ----------------------------------------------------
        # Prediction probabilities
        # ----------------------------------------------------

        probabilities = (
            model_pipeline.predict_proba(
                input_df
            )[0]
        )


        classes = model_pipeline.classes_


        probability_df = pd.DataFrame({

            "Outcome": classes,

            "Probability": probabilities

        })


        probability_df[
            "Probability (%)"
        ] = (
            probability_df[
                "Probability"
            ] * 100
        ).round(2)


        # ----------------------------------------------------
        # RESULT
        # ----------------------------------------------------

        st.divider()

        st.subheader(
            "🎯 Prediction Result"
        )


        if student_name.strip():

            st.write(
                f"**Student:** {student_name}"
            )


        st.markdown(
            f"""
            <div class="prediction-box">

            <div>Predicted Student Outcome</div>

            <div class="prediction-result">
            {prediction}
            </div>

            </div>
            """,
            unsafe_allow_html=True
        )


        # ----------------------------------------------------
        # PROBABILITIES
        # ----------------------------------------------------

        st.subheader(
            "📈 Prediction Probabilities"
        )


        result_col1, result_col2 = st.columns(2)


        with result_col1:

            st.dataframe(

                probability_df[
                    [
                        "Outcome",
                        "Probability (%)"
                    ]
                ],

                use_container_width=True,

                hide_index=True

            )


        with result_col2:

            chart_data = (
                probability_df
                .set_index("Outcome")
                ["Probability (%)"]
            )

            st.bar_chart(
                chart_data
            )


        # ----------------------------------------------------
        # MAIN RESULT MESSAGE
        # ----------------------------------------------------

        highest_probability = (
            probabilities.max() * 100
        )


        st.success(

            f"The model predicts **{prediction}** "
            f"as the most likely student outcome, "
            f"with an estimated probability of "
            f"**{highest_probability:.2f}%**."

        )


    except Exception as error:

        st.error(
            "❌ An error occurred while processing "
            "the student's information."
        )

        st.exception(error)


# ============================================================
# MODEL EVALUATION
# ============================================================

with st.expander(
    "📊 View Model Evaluation"
):

    st.subheader(
        "Test Set Performance"
    )


    col1, col2, col3, col4 = st.columns(4)


    with col1:

        st.metric(
            "Accuracy",
            f"{accuracy:.2%}"
        )


    with col2:

        st.metric(
            "Precision",
            f"{precision:.2%}"
        )


    with col3:

        st.metric(
            "Recall",
            f"{recall:.2%}"
        )


    with col4:

        st.metric(
            "F1 Score",
            f"{f1:.2%}"
        )


    # --------------------------------------------------------
    # Confusion Matrix
    # --------------------------------------------------------

    st.subheader(
        "Confusion Matrix"
    )


    fig, ax = plt.subplots()

    ax.imshow(
        confusion
    )


    ax.set_xlabel(
        "Predicted Label"
    )

    ax.set_ylabel(
        "Actual Label"
    )

    ax.set_title(
        "Logistic Regression Confusion Matrix"
    )


    ax.set_xticks(
        range(
            len(
                model_pipeline.classes_
            )
        )
    )

    ax.set_yticks(
        range(
            len(
                model_pipeline.classes_
            )
        )
    )


    ax.set_xticklabels(
        model_pipeline.classes_
    )

    ax.set_yticklabels(
        model_pipeline.classes_
    )


    for i in range(
        confusion.shape[0]
    ):

        for j in range(
            confusion.shape[1]
        ):

            ax.text(

                j,

                i,

                confusion[i, j],

                ha="center",

                va="center"

            )


    st.pyplot(fig)

    plt.close(fig)


    # --------------------------------------------------------
    # Classification Report
    # --------------------------------------------------------

    st.subheader(
        "Classification Report"
    )


    st.text(
        report
    )


# ============================================================
# DATASET INFORMATION
# ============================================================

with st.expander(
    "📋 Dataset Information"
):

    info_col1, info_col2, info_col3 = st.columns(3)


    with info_col1:

        st.metric(
            "Students",
            len(df)
        )


    with info_col2:

        st.metric(
            "Predictor Features",
            len(X.columns)
        )


    with info_col3:

        st.metric(
            "Missing Values",
            int(
                df.isnull()
                .sum()
                .sum()
            )
        )


    st.subheader(
        "Outcome Distribution"
    )


    st.bar_chart(
        df["Target"].value_counts()
    )


# ============================================================
# FOOTER
# ============================================================

st.divider()

st.caption(
    "🎓 Student Performance Predictor • "
    "Logistic Regression • "
    "Model output should be interpreted as a prediction, "
    "not a guarantee."
)

Overwriting /content/drive/MyDrive/student-performance-app/app.py


In [6]:
import os

APP_PATH = "/content/drive/MyDrive/student-performance-app/app.py"

print(
    "app.py exists:",
    os.path.exists(APP_PATH)
)

print(
    "data.csv exists:",
    os.path.exists(
        "/content/drive/MyDrive/student-performance-app/data.csv"
    )
)

app.py exists: True
data.csv exists: True


In [7]:
import subprocess
import time
import re
from IPython.display import display, HTML

APP_PATH = "/content/drive/MyDrive/student-performance-app/app.py"

# Stop previous instances
!pkill -f "streamlit run" 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true

# Start Streamlit
streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        APP_PATH,
        "--server.address=0.0.0.0",
        "--server.port=8501",
        "--server.headless=true",
        "--browser.gatherUsageStats=false"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Give Streamlit time to start
time.sleep(5)

# Start Cloudflare tunnel
cloudflared_process = subprocess.Popen(
    [
        "/content/cloudflared",
        "tunnel",
        "--url",
        "http://localhost:8501"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None

for _ in range(30):

    line = cloudflared_process.stdout.readline()

    if line:

        print(line.strip())

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            line
        )

        if match:

            public_url = match.group(0)

            break

    time.sleep(1)


if public_url:

    print("\n✅ STREAMLIT APP IS RUNNING\n")

    print(public_url)

    display(
        HTML(
            f"""
            <h2>🎓 Student Performance Predictor</h2>

            <p>
                <a href="{public_url}"
                   target="_blank"
                   style="
                       font-size:22px;
                       text-decoration:none;
                   ">

                   🚀 OPEN MY STREAMLIT APP

                </a>
            </p>
            """
        )
    )

else:

    print(
        "❌ Could not create the public URL."
    )

^C
^C
2026-10-01T08:45:57Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-01T08:45:57Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-01T08:46:01Z INF +--------------------------------------------------------------------------------------------+
2026-10-01T08:46:01Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-01T08:46:01Z INF |  https://drunk-specialties-hebrew-bridges.tryclo